# Fine-tuning ByT5 pour normaliser le wolof

Ce notebook entraîne `google/byt5-small` à passer d'un wolof écrit « à la française / SMS » à l'orthographe officielle.

Données : [Lahad/wolof-normalization-pairs](https://huggingface.co/datasets/Lahad/wolof-normalization-pairs) (31 752 paires d'entraînement, 200 de validation, 688 de test, CC BY-SA 4.0).

**Avant de commencer :** *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Exécution → Tout exécuter*.

Durée indicative : ~1 h sur T4, ~20 min sur L4/A100.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code et dépendances

In [ ]:
%cd /content
!test -d wolof-normalizer || git clone -q https://github.com/LahadMbacke/wolof-normalizer.git
%cd /content/wolof-normalizer
!git pull -q
!pip install -q -e ".[eval]" "transformers==5.20.2" accelerate datasets

## 2. Les données (depuis Hugging Face)

In [ ]:
from datasets import load_dataset

DATA_REPO = "Lahad/wolof-normalization-pairs"
ds = load_dataset(DATA_REPO)
print(ds)
for row in ds["train"].select(range(5)):
    print(f"{row['origin']:>9} | {row['source']}\n          → {row['target']}\n")

## 3. (Optionnel) Sauvegarder sur Google Drive

Colab peut couper la session : en enregistrant les checkpoints sur Drive, on ne perd pas l'entraînement. Mettre `USE_DRIVE = False` pour s'en passer.

In [ ]:
USE_DRIVE = True
OUTPUT_DIR = "runs/byt5-small"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/wolof-normalizer/byt5-small"
print(OUTPUT_DIR)

## 4. (Optionnel) Connexion Hugging Face pour publier le modèle

Renseigner `HUB_MODEL_ID` (ex. `Lahad/wolof-normalizer-byt5-small`) et un token *write*. Laisser `None` pour ne rien publier.

In [ ]:
HUB_MODEL_ID = None  # "Lahad/wolof-normalizer-byt5-small"
if HUB_MODEL_ID:
    from huggingface_hub import notebook_login
    notebook_login()

## 5. Entraînement

Si la session Colab est coupée, relancer les cellules 1 à 5 : l'entraînement reprend au dernier checkpoint enregistré sur Drive. Le dossier `best` n'est créé qu'à la fin.

In [ ]:
push = f"--push-to-hub {HUB_MODEL_ID}" if HUB_MODEL_ID else ""
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/train_byt5.py --data-repo {DATA_REPO} --output-dir "{OUTPUT_DIR}" --epochs 5 --batch-size 8 --grad-accum 4 {push}

In [ ]:
!ls "{OUTPUT_DIR}"  # doit contenir « best » une fois l'entraînement terminé

## 6. Évaluation sur le test (688 vraies paires jamais vues)

In [ ]:
!python scripts/run_evaluation.py --data-repo {DATA_REPO} --model "{OUTPUT_DIR}/best"

## 7. Essayer

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(f"{OUTPUT_DIR}/best")
model = AutoModelForSeq2SeqLM.from_pretrained(f"{OUTPUT_DIR}/best").to("cuda").eval()

def normalize(sentences):
    batch = tokenizer(sentences, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        out = model.generate(**batch, max_length=512, num_beams=4)
    return tokenizer.batch_decode(out, skip_special_tokens=True)

examples = [
    "Nanga def ? Mangui fi rek, dieuredieuf",
    "dafa beug wakh ak yaw thi kanam",
    "gnou bari dagnou meuna dem keur gui souba",
]
for s, n in zip(examples, normalize(examples)):
    print(s, "->", n)

## 8. Télécharger le modèle (si pas sur Drive ni Hugging Face)

In [ ]:
!cd "{OUTPUT_DIR}" && zip -qr /content/byt5-wolof-best.zip best
from google.colab import files
files.download("/content/byt5-wolof-best.zip")